# Snowflake x GenAI: LLM Functions: Lesson 6: The SUMMARIZE() Function

Views over the extracted songs, SUMMARIZE and SENTIMENT, bulk processing from a stage folder.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `SXGA2_LLM_ADMIN` (ACCOUNTADMIN), `SXGA2_LLM_ROLE` (SYSADMIN) and `SXGA2_LLM_WH` (COMPUTE_WH).
- Course code that no longer works runs **as written** and reports its error; a **Substitute** cell with current functions follows it.
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- 🛠️ **Course update needed** marks what has to change in the course itself.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "SXGA2_LLM_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "SXGA2_LLM_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "SXGA2_LLM_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: Views for Exploring the SUMMARIZE() Function

> **Course:** 🥋 LAB: Convert the Select to a View (LLMM_180): `create view doc_ai_db.doc_ai_schema.parsed_song_data as` over the scored select, as DOC_AI_ROLE.

In [ ]:
%%sql -r create_parsed_song_data_as_given
-- From screenshot LLMM_180 (the course's scored select)
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
create view doc_ai_db.doc_ai_schema.parsed_song_data as
select json:__documentMetadata:ocrScore as overall_confidence_score
    , json:song_title[0].score as song_title_score
    , json:song_title[0].value::text as song_title
    , json:performing_artist[0].score as perf_art_score
    , json:performing_artist[0].value::text as performing_artist
    , json:lyrics[0].score as lyrics_score
    , json:lyrics[0].value::text as song_lyrics
    , json:authors[0].score as authors_score
    , json:authors[0].value::text as authors
    , json:lyrics_copyright_holders[0].score as copyright_score
    , json:lyrics_copyright_holders[0].value::text as copyright_holders
    , json:lyric_licensor_provider[0].score as provider_score
    , json:lyric_licensor_provider[0].value::text as licensor_provider
    , json:source[0].score as source_score
    , json:source[0].value::text as source
  from sxga2.lyrics.raw_songs;

In [ ]:
%%sql -r create_parsed_song_data_substitute
-- Substitute: the same view name and value columns over the AI_EXTRACT output (the score columns are dropped)
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
create or replace view doc_ai_db.doc_ai_schema.parsed_song_data as
select json:response:song_title::text as song_title
    , json:response:performing_artist::text as performing_artist
    , json:response:lyrics::text as song_lyrics
    , json:response:authors::text as authors
    , json:response:lyrics_copyright_holders::text as copyright_holders
    , json:response:lyric_licensor_provider::text as licensor_provider
    , json:response:source::text as source
  from sxga2.lyrics.raw_songs;

> **Course:** 🥋 LAB: Give SYSADMIN Access to the View (LLMM_182, 183): USAGE on DOC_AI_DB and its schema, SELECT on the view.

In [ ]:
%%sql -r grant_sysadmin_view_access
-- SQL equivalent of the Privileges dialogs (LLMM_182 shows the learner as ACCOUNTADMIN, the owner of DOC_AI_DB). Real ACCOUNTADMIN.
use role accountadmin;
grant usage on database doc_ai_db to role {{SYSADMIN_ROLE}};
grant usage on schema doc_ai_db.doc_ai_schema to role {{SYSADMIN_ROLE}};
grant select on view doc_ai_db.doc_ai_schema.parsed_song_data to role {{SYSADMIN_ROLE}};

> ⚠️ **Audit note:** In the course DOC_AI_DB is owned by ACCOUNTADMIN, who makes this grant (LLMM_182); DOC_AI_ROLE only received CREATE SCHEMA/MODIFY/USAGE, so it cannot pass USAGE on to another role. The cell therefore uses the real ACCOUNTADMIN.

> **Course:** 📓 Differences in the Views: create a view for SYSADMIN that focuses on the data values, adding CORTEX.SUMMARIZE() and CORTEX.SENTIMENT().

In [ ]:
%%sql -r song_summaries_as_given
use role {{SYSADMIN_ROLE}};
use warehouse {{WAREHOUSE}};
execute immediate 'begin
  execute immediate ''create view sxga2.lyrics.song_summaries as
select song_title
    , snowflake.cortex.summarize(song_lyrics) as song_summary
    , snowflake.cortex.sentiment(song_lyrics) as song_sentiment
from DOC_AI_DB.DOC_AI_SCHEMA.PARSED_SONG_DATA'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r song_summaries_substitute
-- Substitute: ai_summarize and ai_sentiment (a label instead of a number)
use role {{SYSADMIN_ROLE}};
use warehouse {{WAREHOUSE}};
create or replace view sxga2.lyrics.song_summaries as
select song_title
    , ai_summarize(song_lyrics) as song_summary
    , ai_sentiment(song_lyrics):categories[0]:sentiment::text as song_sentiment
from DOC_AI_DB.DOC_AI_SCHEMA.PARSED_SONG_DATA;

In [ ]:
%%sql -r select_song_summaries
select * from sxga2.lyrics.song_summaries;

> **Course:** 🎯 CHALLENGE LAB: Add & Process More Files. Upload the training songs to the stage (in a folder), then edit the insert statement (add the folder path) so each file is processed. Hints: you may need to change roles; use LS to list the files; one insert per file, including the directory in the file name.

In [ ]:
%%sql -r load_training_folder
-- SQL equivalent of uploading to a folder named training (LLMM_185, 188)
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
copy files into @doc_ai_db.doc_ai_schema.lyrics_pdfs/training/
from @UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS
files = ('Mary Had A Little Lamb by Storybooksong.pdf', 'Beyond the Sea by Bobby Darin.pdf', 'September by Earth, Wind & Fire.pdf', 'Gloria by Laura Branigan.pdf', 'Fortnight by Taylor Swift.pdf', 'Cutie Pie by One Way.pdf');
alter stage lyrics_pdfs refresh;
ls @lyrics_pdfs/training;

In [ ]:
%%sql -r insert_training_folder
-- Derived solution (LLMM_187): the insert with the folder in the path
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
insert into sxga2.lyrics.raw_songs
select lyrics_extractor(to_file('@doc_ai_db.doc_ai_schema.lyrics_pdfs', relative_path))
from directory(@lyrics_pdfs)
where relative_path like 'training/%';

In [ ]:
%%sql -r training_songs_summaries
use role {{SYSADMIN_ROLE}};
use warehouse {{WAREHOUSE}};
select * from sxga2.lyrics.song_summaries;

## Page: Automating Processing & Testing Translation

> **Course:** 🎯 CHALLENGE LAB: Load the Final Set of PDFs: download and unzip More_songs.zip; upload the files into a new directory in your stage called "automated" (LLMM_189).

In [ ]:
%%sql -r load_automated_folder
-- SQL equivalent of the upload
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
copy files into @doc_ai_db.doc_ai_schema.lyrics_pdfs/automated/
from @UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS
files = ('99 Luftballons by Nena.pdf', 'Blood Sweat & Tears by BTS JA.pdf', 'Blood Sweat & Tears by BTS KO.pdf', 'Flashdance... vilken känsla by Ingmar Nordströms.pdf', 'Garota de Ipanema by Antonio Carlos Jobim.pdf', 'Gloria by Umberto Tozzi.pdf', 'La Mer by Charles Trénet.pdf', '_3 by Bad Bunny.pdf');
alter stage lyrics_pdfs refresh;

> **Course:** 🥋 LAB: Run the Statements to See How RELATIVE_PATH can be used to bulk process the files.

In [ ]:
%%sql -r relative_path_statements
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
-- run the list command on our newest set of files 
ls @lyrics_pdfs/automated;

--view the relative_path of all files in the stage
select relative_path
from directory(@lyrics_pdfs);

--view the relative path of ONLY the files in a particular directony
select relative_path
from directory(@lyrics_pdfs)
where relative_path like '%automated%';

> **Course:** 🥋 LAB: Process All Files in the Automated Folder (LLMM_191).

In [ ]:
%%sql -r process_automated_as_given
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
execute immediate 'begin
  execute immediate ''insert into sxga2.lyrics.raw_songs
SELECT DOC_AI_DB.DOC_AI_SCHEMA.LYRICS_EXTRACTOR!PREDICT(
  GET_PRESIGNED_URL(@LYRICS_PDFS, relative_path), 1)
  from directory(@lyrics_pdfs)
  where relative_path like ''''%automated%'''''';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r process_automated_substitute
-- Substitute: AI_EXTRACT per file in the folder
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
insert into sxga2.lyrics.raw_songs
select lyrics_extractor(to_file('@doc_ai_db.doc_ai_schema.lyrics_pdfs', relative_path))
from directory(@lyrics_pdfs)
where relative_path like '%automated%';

> **Course:** 🥋 LAB: View Summaries for the Latest Batch of Songs (LLMM_193-195): compare the summaries and sentiment of songs in two languages.

In [ ]:
%%sql -r latest_batch_summaries
use role {{SYSADMIN_ROLE}};
use warehouse {{WAREHOUSE}};
select * from sxga2.lyrics.song_summaries;

In [ ]:
%%sql -r raw_songs_total
select count(*) as rows_in_raw_songs from sxga2.lyrics.raw_songs;

> 🛠️ **Course update needed:** Sentiment: the course compares numeric scores between a song and its translation ("quantify a significant change"). With labels that comparison needs a different method (for example AI_COMPLETE scoring, or label counts per song).